# Stochastic Service Model Examples

This notebook demonstrates how **stochastic demand and lead-time uncertainty** affect inventory performance through Monte Carlo simulation.

You will learn to:
- Simulate daily inventory cycles with random demand and lead times
- Measure **realised** cycle service level and fill rate from simulation
- Compare analytical safety-stock formulas against simulated outcomes
- Explore how different sources of variability drive inventory requirements

Each example uses the `SafetyStockCalculator` from this repository to set the reorder point analytically, then runs a stochastic simulation to verify the result.

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
from statistics import NormalDist
from typing import Literal, Optional
import math
import random
import sys


_loaded_from = None

# 1) Try normal import first (works if package is installed in kernel env)
try:
    from safety_stock.calculator import SafetyStockCalculator
    from safety_stock.models import DemandLeadTimeProfile, InputScenario, ServiceTarget
    _loaded_from = "installed package"
except Exception:
    # 2) Try local source import from current working tree
    repo_root = None
    start = Path.cwd()
    for root in [start, *start.parents]:
        src_pkg = root / "src" / "safety_stock"
        if (src_pkg / "calculator.py").exists() and (src_pkg / "models.py").exists():
            repo_root = root
            break

    if repo_root is not None:
        src_dir = repo_root / "src"
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))

        for name in list(sys.modules):
            if name == "safety_stock" or name.startswith("safety_stock."):
                del sys.modules[name]

        from safety_stock.calculator import SafetyStockCalculator
        from safety_stock.models import DemandLeadTimeProfile, InputScenario, ServiceTarget
        _loaded_from = f"local source at {repo_root}"
    else:
        # 3) Fallback: inline implementation for virtual GitHub FS environments
        ServiceTargetKind = Literal["cycle_service_level", "fill_rate"]

        @dataclass(frozen=True)
        class DemandLeadTimeProfile:
            mean_demand_per_period: float
            stdev_demand_per_period: float
            mean_lead_time_periods: float
            stdev_lead_time_periods: float = 0.0
            review_period_periods: float = 0.0

            def validate(self) -> None:
                if self.mean_demand_per_period <= 0:
                    raise ValueError("mean_demand_per_period must be > 0")
                if self.stdev_demand_per_period < 0:
                    raise ValueError("stdev_demand_per_period must be >= 0")
                if self.mean_lead_time_periods <= 0:
                    raise ValueError("mean_lead_time_periods must be > 0")
                if self.stdev_lead_time_periods < 0:
                    raise ValueError("stdev_lead_time_periods must be >= 0")
                if self.review_period_periods < 0:
                    raise ValueError("review_period_periods must be >= 0")

        @dataclass(frozen=True)
        class ServiceTarget:
            kind: ServiceTargetKind
            value: float
            order_quantity: Optional[float] = None

            def validate(self) -> None:
                if not (0 < self.value < 1):
                    raise ValueError("Service target value must be between 0 and 1")
                if self.kind == "fill_rate":
                    if self.order_quantity is None or self.order_quantity <= 0:
                        raise ValueError("fill_rate target requires order_quantity > 0")

        @dataclass(frozen=True)
        class InputScenario:
            sku: str
            profile: DemandLeadTimeProfile
            target: ServiceTarget

            def validate(self) -> None:
                if not self.sku.strip():
                    raise ValueError("sku must not be empty")
                self.profile.validate()
                self.target.validate()

        @dataclass(frozen=True)
        class CalculationResult:
            sku: str
            target_kind: ServiceTargetKind
            target_value: float
            z_value: float
            expected_demand_in_protection_period: float
            stdev_demand_in_protection_period: float
            safety_stock: float
            reorder_point: float
            expected_shortage_per_cycle: Optional[float] = None

        class SafetyStockCalculator:
            _normal = NormalDist(mu=0.0, sigma=1.0)

            def calculate(self, scenario: InputScenario) -> CalculationResult:
                scenario.validate()

                mean_pp = self._protection_period_mean(scenario)
                stdev_pp = self._protection_period_stdev(scenario)
                target = scenario.target

                if stdev_pp == 0:
                    z_value = 0.0
                    safety_stock = 0.0
                    expected_shortage = 0.0 if target.kind == "fill_rate" else None
                elif target.kind == "cycle_service_level":
                    z_value = self._normal.inv_cdf(target.value)
                    safety_stock = z_value * stdev_pp
                    expected_shortage = None
                else:
                    order_qty = float(target.order_quantity)
                    z_value = self._solve_z_for_fill_rate(target.value, order_qty, stdev_pp)
                    safety_stock = z_value * stdev_pp
                    expected_shortage = self._expected_shortage_per_cycle(z_value, stdev_pp)

                reorder_point = mean_pp + safety_stock

                return CalculationResult(
                    sku=scenario.sku,
                    target_kind=target.kind,
                    target_value=target.value,
                    z_value=z_value,
                    expected_demand_in_protection_period=mean_pp,
                    stdev_demand_in_protection_period=stdev_pp,
                    safety_stock=safety_stock,
                    reorder_point=reorder_point,
                    expected_shortage_per_cycle=expected_shortage,
                )

            @staticmethod
            def _protection_period_mean(scenario: InputScenario) -> float:
                p = scenario.profile
                return p.mean_demand_per_period * (p.mean_lead_time_periods + p.review_period_periods)

            @staticmethod
            def _protection_period_stdev(scenario: InputScenario) -> float:
                p = scenario.profile
                protection_window = p.mean_lead_time_periods + p.review_period_periods
                demand_component = protection_window * (p.stdev_demand_per_period ** 2)
                lead_time_component = (p.mean_demand_per_period ** 2) * (p.stdev_lead_time_periods ** 2)
                variance = demand_component + lead_time_component
                return math.sqrt(variance)

            def _solve_z_for_fill_rate(self, fill_rate: float, order_qty: float, sigma_pp: float) -> float:
                target_shortage = (1 - fill_rate) * order_qty
                lo, hi = -4.0, 8.0
                for _ in range(80):
                    mid = 0.5 * (lo + hi)
                    shortage = self._expected_shortage_per_cycle(mid, sigma_pp)
                    if shortage > target_shortage:
                        lo = mid
                    else:
                        hi = mid
                return 0.5 * (lo + hi)

            def _expected_shortage_per_cycle(self, z_value: float, sigma_pp: float) -> float:
                cdf = self._normal.cdf(z_value)
                pdf = self._normal.pdf(z_value)
                loss = pdf - z_value * (1 - cdf)
                return sigma_pp * loss

        _loaded_from = "inline fallback implementation"


calculator = SafetyStockCalculator()
print(f"Calculator ready ({_loaded_from}).")

## Helper Utilities

The `show_result` function prints analytical outputs. The `simulate_inventory` function runs a
Monte Carlo (Q, R) continuous-review simulation to measure realised service performance.

In [ ]:
def show_result(result):
    """Print the analytical calculation result."""
    data = asdict(result)
    print(f"SKU: {data['sku']}")
    print(f"Target type: {data['target_kind']}")
    print(f"Target value: {data['target_value']:.3f}")
    print(f"z-value: {data['z_value']:.3f}")
    print(f"Mean demand in protection period: {data['expected_demand_in_protection_period']:.2f}")
    print(f"Std dev in protection period: {data['stdev_demand_in_protection_period']:.2f}")
    print(f"Safety stock: {data['safety_stock']:.2f}")
    print(f"Reorder point: {data['reorder_point']:.2f}")
    if data['expected_shortage_per_cycle'] is not None:
        print(f"Expected shortage per cycle: {data['expected_shortage_per_cycle']:.2f}")


def simulate_inventory(
    mean_demand: float,
    stdev_demand: float,
    mean_lead_time: float,
    stdev_lead_time: float,
    reorder_point: float,
    order_quantity: float,
    num_periods: int = 10_000,
    seed: int = 42,
):
    """Simulate a (Q, R) continuous-review inventory system.

    Returns a dict with realised cycle service level, fill rate, and
    summary statistics.

    The simulation tracks:
    - inventory_position (on-hand + on-order - backorders)
    - on_hand inventory
    - outstanding orders arriving at future periods
    """
    rng = random.Random(seed)

    on_hand = reorder_point  # start at reorder point
    inventory_position = on_hand
    pending_orders = {}  # period -> quantity arriving

    total_demand = 0.0
    total_fulfilled = 0.0
    cycles = 0
    stockout_cycles = 0

    for t in range(num_periods):
        # Receive any pending orders arriving this period
        if t in pending_orders:
            on_hand += pending_orders.pop(t)

        # Generate demand (non-negative)
        demand = max(0.0, rng.gauss(mean_demand, stdev_demand))
        total_demand += demand

        # Fulfil demand from on-hand
        fulfilled = min(on_hand, demand)
        total_fulfilled += fulfilled
        on_hand -= fulfilled

        # Track stockouts at cycle level
        if demand > fulfilled + 1e-9:
            stockout_cycles += 1

        # Check reorder point — place order when inventory_position <= R
        inventory_position = on_hand + sum(pending_orders.values())
        if inventory_position <= reorder_point:
            lead_time = max(1, round(rng.gauss(mean_lead_time, stdev_lead_time)))
            arrival = t + lead_time
            pending_orders[arrival] = pending_orders.get(arrival, 0) + order_quantity
            inventory_position += order_quantity
            cycles += 1

    realised_csl = 1.0 - stockout_cycles / max(num_periods, 1)
    realised_fill_rate = total_fulfilled / max(total_demand, 1e-9)

    return {
        "num_periods": num_periods,
        "total_cycles": cycles,
        "stockout_periods": stockout_cycles,
        "realised_cycle_service_level": realised_csl,
        "realised_fill_rate": realised_fill_rate,
        "total_demand": total_demand,
        "total_fulfilled": total_fulfilled,
    }


def print_sim_results(sim):
    """Pretty-print simulation results."""
    print(f"Periods simulated: {sim['num_periods']:,}")
    print(f"Replenishment cycles: {sim['total_cycles']:,}")
    print(f"Stockout periods: {sim['stockout_periods']:,}")
    print(f"Realised cycle service level: {sim['realised_cycle_service_level']:.4f}")
    print(f"Realised fill rate: {sim['realised_fill_rate']:.4f}")

---

## Example 1: Demand-Only Uncertainty (Cycle Service Level)

In this example, lead time is **deterministic** (σ_L = 0). All uncertainty comes from random
demand each period.

We compute the analytical reorder point for a 95% cycle service level, then simulate to verify.

In [ ]:
scenario_demand_only = InputScenario(
    sku="DEMAND-ONLY",
    profile=DemandLeadTimeProfile(
        mean_demand_per_period=100,
        stdev_demand_per_period=30,
        mean_lead_time_periods=3.0,
        stdev_lead_time_periods=0.0,  # deterministic lead time
    ),
    target=ServiceTarget(kind="cycle_service_level", value=0.95),
)

result = calculator.calculate(scenario_demand_only)
print("=== Analytical Result ===")
show_result(result)

print("\n=== Monte Carlo Simulation ===")
sim = simulate_inventory(
    mean_demand=100, stdev_demand=30,
    mean_lead_time=3, stdev_lead_time=0.0,
    reorder_point=result.reorder_point,
    order_quantity=300,  # Q = 3 periods of mean demand
    num_periods=20_000,
)
print_sim_results(sim)
print(f"\nTarget CSL: {result.target_value:.2%}  |  Realised CSL: {sim['realised_cycle_service_level']:.2%}")

---

## Example 2: Demand + Lead-Time Uncertainty

Now we add **stochastic lead times** (σ_L > 0). The analytical formula accounts for this
through the combined protection-period variance:

```
σ_T = √(T × σ_D² + μ_D² × σ_L²)
```

Notice how adding lead-time variability increases the required safety stock.

In [ ]:
scenario_both = InputScenario(
    sku="DEMAND-AND-LT",
    profile=DemandLeadTimeProfile(
        mean_demand_per_period=100,
        stdev_demand_per_period=30,
        mean_lead_time_periods=3.0,
        stdev_lead_time_periods=1.0,  # stochastic lead time
    ),
    target=ServiceTarget(kind="cycle_service_level", value=0.95),
)

result_both = calculator.calculate(scenario_both)
print("=== Analytical Result ===")
show_result(result_both)

print("\n=== Monte Carlo Simulation ===")
sim_both = simulate_inventory(
    mean_demand=100, stdev_demand=30,
    mean_lead_time=3, stdev_lead_time=1.0,
    reorder_point=result_both.reorder_point,
    order_quantity=300,
    num_periods=20_000,
)
print_sim_results(sim_both)

print("\n=== Comparison ===")
print(f"Safety stock (demand only): {result.safety_stock:.2f}")
print(f"Safety stock (demand + LT): {result_both.safety_stock:.2f}")
print(f"Increase from lead-time uncertainty: {result_both.safety_stock - result.safety_stock:.2f} units "
      f"(+{(result_both.safety_stock / result.safety_stock - 1) * 100:.1f}%)")

---

## Example 3: Fill Rate Service Model

The **fill rate** (β) measures the fraction of demand fulfilled immediately from stock.
Unlike cycle service level, it depends on the **order quantity** Q.

We target a 98% fill rate and compare the z-value with the 95% CSL example above.

In [ ]:
scenario_fill = InputScenario(
    sku="FILL-RATE-98",
    profile=DemandLeadTimeProfile(
        mean_demand_per_period=100,
        stdev_demand_per_period=30,
        mean_lead_time_periods=3.0,
        stdev_lead_time_periods=1.0,
    ),
    target=ServiceTarget(kind="fill_rate", value=0.98, order_quantity=300),
)

result_fill = calculator.calculate(scenario_fill)
print("=== Analytical Result ===")
show_result(result_fill)

print("\n=== Monte Carlo Simulation ===")
sim_fill = simulate_inventory(
    mean_demand=100, stdev_demand=30,
    mean_lead_time=3, stdev_lead_time=1.0,
    reorder_point=result_fill.reorder_point,
    order_quantity=300,
    num_periods=20_000,
)
print_sim_results(sim_fill)

print("\n=== CSL vs Fill Rate Comparison ===")
print(f"95% CSL  → z = {result_both.z_value:.3f}, SS = {result_both.safety_stock:.2f}")
print(f"98% Fill → z = {result_fill.z_value:.3f}, SS = {result_fill.safety_stock:.2f}")
print(f"\nFill rate typically requires less safety stock than an equivalent CSL target\n"
      f"because it accounts for the order quantity absorbing some shortage.")

---

## Example 4: Periodic Review System

In a **periodic-review** system, inventory is checked every P periods instead of continuously.
This increases the protection period from μ_L to μ_L + P, requiring more safety stock.

Compare a continuous-review (P = 0) vs weekly review (P = 5 days) system.

In [ ]:
profile_base = dict(
    mean_demand_per_period=50,
    stdev_demand_per_period=12,
    mean_lead_time_periods=4.0,
    stdev_lead_time_periods=0.5,
)

# Continuous review (P = 0)
scenario_cont = InputScenario(
    sku="CONTINUOUS",
    profile=DemandLeadTimeProfile(**profile_base, review_period_periods=0.0),
    target=ServiceTarget(kind="cycle_service_level", value=0.95),
)

# Periodic review (P = 5)
scenario_periodic = InputScenario(
    sku="PERIODIC-P5",
    profile=DemandLeadTimeProfile(**profile_base, review_period_periods=5.0),
    target=ServiceTarget(kind="cycle_service_level", value=0.95),
)

result_cont = calculator.calculate(scenario_cont)
result_periodic = calculator.calculate(scenario_periodic)

print("=== Continuous Review (P = 0) ===")
show_result(result_cont)
print()
print("=== Periodic Review (P = 5) ===")
show_result(result_periodic)
print()
print(f"Safety stock increase from periodic review: "
      f"{result_periodic.safety_stock - result_cont.safety_stock:.2f} units "
      f"(+{(result_periodic.safety_stock / result_cont.safety_stock - 1) * 100:.1f}%)")
print(f"Reorder point increase: "
      f"{result_periodic.reorder_point - result_cont.reorder_point:.2f} units")

---

## Example 5: Sensitivity — Lead-Time Variability Impact

How much does lead-time variability (σ_L) contribute to safety stock?

We scan σ_L from 0 (deterministic) to 2.0 periods and observe the growth in safety stock,
verifying each point with a simulation.

In [ ]:
lt_stdevs = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0]

print(f"{'σ_L':>6} {'σ_T':>10} {'SS (analytical)':>16} {'ROP':>10} {'Sim CSL':>10}")
print("-" * 56)

for lt_std in lt_stdevs:
    scenario = InputScenario(
        sku=f"LT-{lt_std:.2f}",
        profile=DemandLeadTimeProfile(
            mean_demand_per_period=100,
            stdev_demand_per_period=30,
            mean_lead_time_periods=3.0,
            stdev_lead_time_periods=lt_std,
        ),
        target=ServiceTarget(kind="cycle_service_level", value=0.95),
    )
    res = calculator.calculate(scenario)

    sim = simulate_inventory(
        mean_demand=100, stdev_demand=30,
        mean_lead_time=3, stdev_lead_time=lt_std,
        reorder_point=res.reorder_point,
        order_quantity=300,
        num_periods=10_000,
    )

    print(f"{lt_std:>6.2f} {res.stdev_demand_in_protection_period:>10.2f} "
          f"{res.safety_stock:>16.2f} {res.reorder_point:>10.2f} "
          f"{sim['realised_cycle_service_level']:>10.4f}")

---

## Example 6: Analytical vs Simulated — Side-by-Side

Run both CSL and fill-rate models across several target levels and compare
the analytical predictions against Monte Carlo simulation results.

In [ ]:
shared_profile = DemandLeadTimeProfile(
    mean_demand_per_period=80,
    stdev_demand_per_period=20,
    mean_lead_time_periods=4.0,
    stdev_lead_time_periods=0.8,
)

Q = 400  # order quantity

# --- Cycle Service Level scan ---
print("=== Cycle Service Level: Analytical vs Simulated ===")
print(f"{'Target':>8} {'z':>8} {'SS':>10} {'ROP':>10} {'Sim CSL':>10} {'Gap':>8}")
print("-" * 58)

for target in [0.85, 0.90, 0.95, 0.97, 0.99]:
    sc = InputScenario(
        sku=f"CSL-{target}",
        profile=shared_profile,
        target=ServiceTarget(kind="cycle_service_level", value=target),
    )
    res = calculator.calculate(sc)
    sim = simulate_inventory(
        mean_demand=80, stdev_demand=20,
        mean_lead_time=4, stdev_lead_time=0.8,
        reorder_point=res.reorder_point,
        order_quantity=Q,
        num_periods=20_000,
    )
    gap = sim['realised_cycle_service_level'] - target
    print(f"{target:>8.2%} {res.z_value:>8.3f} {res.safety_stock:>10.2f} "
          f"{res.reorder_point:>10.2f} {sim['realised_cycle_service_level']:>10.4f} "
          f"{gap:>+8.4f}")

print()

# --- Fill Rate scan ---
print("=== Fill Rate: Analytical vs Simulated ===")
print(f"{'Target':>8} {'z':>8} {'SS':>10} {'ROP':>10} {'Sim FR':>10} {'Gap':>8}")
print("-" * 58)

for target in [0.90, 0.95, 0.97, 0.98, 0.99]:
    sc = InputScenario(
        sku=f"FR-{target}",
        profile=shared_profile,
        target=ServiceTarget(kind="fill_rate", value=target, order_quantity=Q),
    )
    res = calculator.calculate(sc)
    sim = simulate_inventory(
        mean_demand=80, stdev_demand=20,
        mean_lead_time=4, stdev_lead_time=0.8,
        reorder_point=res.reorder_point,
        order_quantity=Q,
        num_periods=20_000,
    )
    gap = sim['realised_fill_rate'] - target
    print(f"{target:>8.2%} {res.z_value:>8.3f} {res.safety_stock:>10.2f} "
          f"{res.reorder_point:>10.2f} {sim['realised_fill_rate']:>10.4f} "
          f"{gap:>+8.4f}")

---

## Example 7: Effect of Order Quantity on Fill Rate

The fill-rate formula depends on the order quantity Q. Larger Q absorbs more demand per cycle,
so the same fill-rate target can be met with **less** safety stock.

We fix β = 0.98 and vary Q to show this relationship.

In [ ]:
fill_profile = DemandLeadTimeProfile(
    mean_demand_per_period=100,
    stdev_demand_per_period=25,
    mean_lead_time_periods=3.0,
    stdev_lead_time_periods=0.7,
)

order_quantities = [100, 200, 300, 400, 600, 800]

print(f"{'Q':>6} {'z':>8} {'SS':>10} {'ROP':>10} {'Sim FR':>10}")
print("-" * 48)

for q in order_quantities:
    sc = InputScenario(
        sku=f"Q-{q}",
        profile=fill_profile,
        target=ServiceTarget(kind="fill_rate", value=0.98, order_quantity=q),
    )
    res = calculator.calculate(sc)
    sim = simulate_inventory(
        mean_demand=100, stdev_demand=25,
        mean_lead_time=3, stdev_lead_time=0.7,
        reorder_point=res.reorder_point,
        order_quantity=q,
        num_periods=10_000,
    )
    print(f"{q:>6} {res.z_value:>8.3f} {res.safety_stock:>10.2f} "
          f"{res.reorder_point:>10.2f} {sim['realised_fill_rate']:>10.4f}")

---

## Student Exercises

1. **Demand variability**: In Example 1, double `stdev_demand_per_period` from 30 to 60.
   How does the safety stock change? What about the simulated CSL?

2. **Deterministic vs stochastic lead time**: Compare Example 1 (σ_L = 0) with Example 2
   (σ_L = 1.0). What percentage of the total safety stock is driven by lead-time uncertainty?

3. **High service level penalty**: In Example 6, note the safety stock for 99% CSL vs 95%.
   How many extra units are needed per percentage point of service?

4. **Fill rate vs CSL**: For the same profile, find the fill-rate target that requires the same
   safety stock as a 95% CSL. (Hint: try different β values in Example 3.)

5. **Order quantity trade-off**: In Example 7, a larger Q reduces safety stock but increases
   average inventory. Compute average inventory as Q/2 + SS for each row. Which Q gives the
   lowest total?

6. **Periodic review cost**: In Example 4, compute how much extra inventory investment the
   periodic review system requires vs continuous review (assume unit cost = \$10).